In [ ]:
# ==========================================================
# SECTION 0: IMPORTS AND DEVICE
# ==========================================================
# Basic numerical + deep learning libraries
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# Select GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================================
# SECTION 3.1: HANKEL MATRIX CONSTRUCTION
# ==========================================================
def hankel_matrix(x, mh):
    """
    Implements lifting operation → converts signal to Hankel matrix.

    Matches Eq. (Hankel construction in Sec 3.1):
    Enhances spatial correlation → induces low-rank structure.
    """
    n = x.shape[0]
    nh = n - mh + 1

    H = torch.zeros((mh, nh), device=x.device)

    for i in range(nh):
        H[:, i] = x[i:i+mh]

    return H

# ==========================================================
# SECTION 3.4: HANKEL INVERSE (Anti-diagonal Averaging)
# ==========================================================
def hankel_inverse(H):
    """
    Implements H^{-1}(·) from Eq. (24).

    Reconstructs 1D signal by averaging along anti-diagonals.
    """
    mh, nh = H.shape

    recon = torch.zeros(mh + nh - 1, device=H.device)
    count = torch.zeros(mh + nh - 1, device=H.device)

    for i in range(mh):
        for j in range(nh):
            recon[i+j] += H[i, j]
            count[i+j] += 1

    return recon / count

# ==========================================================
# SECTION 3.4: AUTOENCODER (LEARNED PROXIMAL OPERATOR)
# ==========================================================
class AED(nn.Module):
    """
    Implements encoder-decoder for adaptive singular value shrinkage.

    Corresponds to:
    Eq. (21) → Encoder
    Eq. (22) → Decoder

    Learns nonlinear mapping instead of fixed SVT.
    """
    def __init__(self, dim):
        super().__init__()

        # Encoder: compress singular values
        self.encoder = nn.Sequential(
            nn.Linear(dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU()
        )

        # Decoder: reconstruct modified singular values
        self.decoder = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, dim)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

# ===============================================
# SECTION 3.4: DURR MODEL (DEEP UNROLLING)
# ===============================================
class DURR(nn.Module):
    """
    Deep Unrolling Rank Reduction Network
    Note:
    K denotes the number of unrolled proximal gradient stages.
    In this implementation, K=2 is chosen for simplicity and faster execution.
    However, K can be varied (e.g., up to 8) to understand the denoising performance
    at the cost of higher computation.
    """
    def __init__(self, K=2, max_sv=32):
        super().__init__()

        self.K = K

        # Learnable step sizes (η_k in paper)
        self.etas = nn.Parameter(torch.ones(K) * 0.5)

        # One AE per stage (θ_k parameters)
        self.aeds = nn.ModuleList([AED(max_sv) for _ in range(K)])

        self.max_sv = max_sv

    def forward(self, Y):
        """
        Input: Hankel matrix Yω
        Output: Denoised matrix Xω
        """
        X = Y.clone()

        for k in range(self.K):

            # --------------------------------------------------
            # Data Consistency Step (Eq. 17)
            # Z = X - η (X - Y)
            # --------------------------------------------------
            Z = X - self.etas[k] * (X - Y)

            # --------------------------------------------------
            # SVD Decomposition (Eq. 18)
            # --------------------------------------------------
            U, S, Vh = torch.linalg.svd(Z, full_matrices=False)
            r = S.shape[0]

            # Pad singular values to fixed size
            S_pad = torch.zeros(self.max_sv, device=S.device)
            S_pad[:r] = S

            # --------------------------------------------------
            # Learned Proximal Operator (AE instead of SVT)
            # --------------------------------------------------
            S_new = self.aeds[k](S_pad.unsqueeze(0)).squeeze(0)
            S_new = S_new[:r]

            # --------------------------------------------------
            # Reconstruction (Eq. 23)
            # --------------------------------------------------
            S_mat = torch.diag(S_new)
            X = U @ S_mat @ Vh

        return X

# ================================================
# FULL PIPELINE (FFT → HANKEL → DURR → IFFT)
# =================================================
def durr_pipeline(Y, model, mh=16):
    """
    Implements full pipeline described in Sec 3:

    1. FFT (Eq. 2)
    2. Per-frequency processing
    3. Hankel lifting
    4. DURR denoising
    5. Inverse Hankel
    6. IFFT (Eq. 25)
    """

    # Transform to frequency domain
    Yf = torch.fft.fft(Y, dim=0)

    Yf_out = torch.zeros_like(Yf, dtype=torch.complex64)

    for k in range(Yf.shape[0]):

        # Extract frequency slice
        y_w = Yf[k, :].real

        # Hankel lifting
        H = hankel_matrix(y_w, mh)

        # DURR processing
        H_denoised = model(H)

        # Inverse Hankel
        y_rec = hankel_inverse(H_denoised)
        y_rec = y_rec[:Y.shape[1]]

        Yf_out[k, :] = y_rec

    # Inverse FFT → time domain reconstruction
    Y_out = torch.fft.ifft(Yf_out, dim=0).real

    return Y_out

# ==========================================================
# SECTION 3.5: LOSS FUNCTION + OPTIMIZER
# ==========================================================
model = DURR().to(device)

# MSE loss (Eq. 27)
criterion = nn.MSELoss()

optimizer = optim.Adam(model.parameters(), lr=1e-3)

# ==========================================================
# SECTION 4: RICKER WAVELET (Synthetic Data Generation)
# ==========================================================
# Synthetic seismic signals use Ricker wavelet
def ricker(f, length=0.128, dt=0.001):
    """
    Generates a Ricker wavelet (used as seismic source).
    Corresponds to Marmousi synthetic setup in the paper.
    """
    t = np.arange(-length/2, length/2, dt)
    return (1 - 2*(np.pi*f*t)**2) * np.exp(-(np.pi*f*t)**2)

# ==========================================================
# SECTION 4.1: SYNTHETIC SEISMIC DATA GENERATION
# ==========================================================
def generate_seismic(n_traces=32, n_samples=128):
    """
    Generates synthetic seismic gather (Y = X + N model in Eq. (1)).

    - Each trace contains shifted wavelets → simulates reflections
    - Produces low-rank structure in Hankel domain (key assumption in paper)
    """
    data = np.zeros((n_samples, n_traces))
    wavelet = ricker(30)  # dominant frequency = 30 Hz

    for i in range(n_traces):
        # Random time shift simulates reflection events
        shift = np.random.randint(10, n_samples // 2)

        # Add wavelet at shifted location
        data[shift:shift+len(wavelet), i] += wavelet[:min(len(wavelet), n_samples-shift)]

    return data

# ==========================================================
# DATASET CREATION (NOISY MODEL: Y = X + N)
# ==========================================================
clean_dataset = []
noisy_dataset = []
for _ in range(4):
    clean = generate_seismic()

    # Add Gaussian noise
    sigma = np.random.uniform(0.1, 0.3)
    noise = np.random.normal(0, sigma, clean.shape)

    noisy = clean + noise

    clean_dataset.append(clean)
    noisy_dataset.append(noisy)

# ===============================
# TRAIN / VAL / TEST SPLIT
# ===============================
indices = np.random.permutation(len(clean_dataset))
clean_dataset = [clean_dataset[i] for i in indices]
noisy_dataset = [noisy_dataset[i] for i in indices]

num_total = len(clean_dataset)

train_end = int(0.7 * num_total)
val_end   = int(0.85 * num_total)

train_clean = clean_dataset[:train_end]
train_noisy = noisy_dataset[:train_end]

val_clean = clean_dataset[train_end:val_end]
val_noisy = noisy_dataset[train_end:val_end]

test_clean = clean_dataset[val_end:]
test_noisy = noisy_dataset[val_end:]
epochs = 50

# ===================
# TRAINING LOOP
# ===================
for epoch in range(epochs):

    model.train()
    train_loss = 0

    for clean, noisy in zip(train_clean, train_noisy):

        clean_t = torch.tensor(clean, dtype=torch.float32).to(device)
        noisy_t = torch.tensor(noisy, dtype=torch.float32).to(device)

        optimizer.zero_grad()

        # Forward pass through full DURR pipeline
        output = durr_pipeline(noisy_t, model)

        # Compute loss
        loss = criterion(output, clean_t)

        # Backpropagation
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_clean)

    # ===================
    # VALIDATION
    # ===================
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for clean, noisy in zip(val_clean, val_noisy):

            clean_t = torch.tensor(clean, dtype=torch.float32).to(device)
            noisy_t = torch.tensor(noisy, dtype=torch.float32).to(device)

            output = durr_pipeline(noisy_t, model)

            loss = criterion(output, clean_t)
            val_loss += loss.item()

    val_loss /= len(val_clean)

    # ===================
    # PRINT EVERY 5 EPOCHS
    # ===================
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:02d}: Train Loss = {train_loss:.6f}, Val Loss = {val_loss:.6f}")
# ========================
# PSNR METRIC Evaluation
# =========================
def compute_psnr(clean, denoised):
    """
    Computes PSNR similar to evaluation metrics section.
    """
    mse = np.mean((clean - denoised)**2)
    max_val = np.max(clean)
    return 10 * np.log10(max_val**2 / mse + 1e-8)

# ==========
# TESTING
# ==========
psnr_list = []

model.eval()

with torch.no_grad():
    for clean, noisy in zip(test_clean, test_noisy):

        clean_t = torch.tensor(clean, dtype=torch.float32).to(device)
        noisy_t = torch.tensor(noisy, dtype=torch.float32).to(device)

        output = durr_pipeline(noisy_t, model)

        output_np = output.cpu().numpy()

        psnr = compute_psnr(clean, output_np)
        psnr_list.append(psnr)

avg_psnr = np.mean(psnr_list)

print("\nFinal Results:")
print(f"Average Test PSNR: {avg_psnr:.2f} dB")

Epoch 01: Train Loss = 0.074310, Val Loss = 0.073952
Epoch 05: Train Loss = 0.074163, Val Loss = 0.073780
Epoch 10: Train Loss = 0.073495, Val Loss = 0.072990
